In [ ]:
import sys, os
sys.path.append("..")
from datetime import datetime
import torch, numpy as np

from src.data import load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

In [ ]:
PAIRS

# Read Historical Data

In [ ]:
PAIRS_ = {
    'Bitcoin': 'XBTEUR',
    'Ethereum': 'ETHEUR',
    'Ripple': 'XRPEUR',
    'Cardano': 'ADAEUR',
    'Solana': 'SOLEUR',
}

In [ ]:
if os.path.exists("../data/historical_data.ptt"):
    print("reading data from file...")
    _data = torch.load("../data/historical_data.ptt")

    times_ = _data['times']
    dt = float(times_.diff().mean().round())
    print(f"dt = {dt}")

    close = _data['close']
    high = _data.get('high', close)
    low = _data.get('low', close)
    volume = _data['volume']

    PAIRS_ = _data['pairs']

elif os.path.exists("../data/Kraken_OHLCVT"):
    print("loading and aligning data from raw files...")
    _data, times = load_and_align_data(PAIRS_, interval=1)

    times_ = torch.tensor([t.timestamp() for t in times], dtype=torch.float64)
    dt = float(times_.diff().mean().round())
    print(f"dt = {dt}")

    close = torch.tensor(get_field(_data, 'close')).T
    high = torch.tensor(get_field(_data, 'high')).T
    low = torch.tensor(get_field(_data, 'low')).T
    volume = torch.tensor(get_field(_data, 'volume')).T
else:
    raise FileNotFoundError("No historical data found. Please download and prepare the data as described in the README.")

In [ ]:
history = [{
    'time': t,
    'close': c,
    'high': h,
    'low': l,
    'volume': v,
} for t, c, h, l, v in zip(times_, close, high, low, volume)]

# history = sorted(history, key=lambda x: x['time'])

len(history)

In [ ]:
from src.environment.proto_v07_discrete import MultiCurrencyEnv

base_t = 60
tau_p = torch.tensor([base_t*20, base_t*60*3, base_t*60*24, base_t*60*24*7], dtype=torch.float32)
print("tau_p:", (tau_p / (3600 * 24)).tolist(), "[days]")

env = MultiCurrencyEnv(
    N=len(PAIRS_),
    C0=1_000,
    tau_p=tau_p,
    tau_p_live=base_t*5,
    size_buckets=(0.50, 1.00),
    bankruptcy_threshold=10.0,
    min_buy_dollars=2.0,
    transaction_eps=1e-2,
    use_dollar_volume=True,
    sell_fee=1.0,
    buy_fee=1.0,
    tax_rate=0.26,
    reward_mode="log",
    val_coeff=1.0,
    roi_coeff=10.0,
    invalid_trade_penalty=0.001,
    done_reward_penalty=100.0,
    save_history=False,
    dtype=torch.float32,
    eps=1e-8,
)
print(f"state_dim: {env.state_dim} - action_dim: {env.action_dim}")

# Create Agent

In [ ]:
class HebbNet:

    def __init__(self, n_in, n_out):
        self.n_in = n_in
        self.n_out = n_out

        self.w_x = torch.nn.Linear(n_in, n_out, bias=True)
        self.w_r = torch.nn.Linear(n_out, n_out, bias=False)

        self.b = torch.nn.Parameter(torch.zeros(self.n_out))

    def reset(self, batch_size=1):
        self.state = torch.zeros([batch_size, self.n_out])

    def __call__(self, x):
        I = self.w_x(x.view(-1, self.n_out)) + self.w_r(self.state)
        if E:
            
        self.state = self.activation(I + self.b)
        return self.state


# Main Training Loop

In [ ]:
loss, rewards, info = [], [], []

In [ ]:
_loss, _rewards, _info = agent.train_on_historical(
    env, history[:int(len(history)*0.9)], n_episodes=10,
    update_interval=32, n_updates=1, burn_in_updates=1,
    max_steps=60000, warm_up=12000,
    lr=3e-4, optim="AdamW", init_optimizer=True,
    max_grad_norm=None,
)
agent.save(f"../data/agent/vaq_{agent.net.recurrent_type}_v07_discrete.ptm")

loss    += _loss
rewards += _rewards
info    += _info

In [ ]:
metric_names = next((list(episode_loss[0].keys()) for episode_loss in loss if len(episode_loss) > 0), [])
if not metric_names:
    raise ValueError("No loss metrics recorded yet.")

ref_metric = metric_names[0]
t_all = np.concatenate([
    np.linspace(i, i + 1, sum(len(np.asarray(loss_dict[ref_metric]).reshape(-1)) for loss_dict in episode_loss), endpoint=False)
    for i, episode_loss in enumerate(loss)
    if len(episode_loss) > 0
])


def flatten_metric(metric_name):
    chunks = []
    for episode_loss in loss:
        if len(episode_loss) == 0:
            continue
        chunks.append(np.concatenate([
            np.asarray(loss_dict[metric_name]).reshape(-1)
            for loss_dict in episode_loss
        ]))
    return np.concatenate(chunks) if chunks else np.array([])


palette = Category10[10]
figs = []
for i, metric_name in enumerate(metric_names):
    values = flatten_metric(metric_name)
    fig = bk.figure(
        title=metric_name.replace("_", " ").title(),
        x_axis_label="Training Iteration [Epochs]",
        y_axis_label="Value",
        width=1000,
        height=260,
    )
    fig.line(t_all[:len(values)], values, line_width=2, legend_label=metric_name, color=palette[i % len(palette)])
    fig.legend.location = "bottom_right"
    figs.append(fig)

bk.show(bk.column(*figs))


In [ ]:
rewards_ = [sum(r) for r in rewards]

fig = bk.figure(title="Total Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Loss", width=900, height=320)
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Total Reward / Episode", color=Category10[10][4])
fig.legend.location = "bottom_right"
bk.show(fig)

rewards_ = [sum(r) / len(r) for r in rewards]

fig = bk.figure(title="Average Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Reward", width=900, height=320)
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.legend.location = "bottom_right"
bk.show(fig)

In [ ]:
episode = -1

ts = [datetime.fromtimestamp(item['t']) for item in info[episode]]
ps = torch.tensor([item['p'] for item in info[episode]])
Vs = torch.tensor([item['V'] for item in info[episode]])
Cs = torch.tensor([item['C'] for item in info[episode]])
vs = torch.tensor([[w * p  / item['V'] for w, p in zip(item['w'], item['p'])] for item in info[episode]])

f0 = bk.figure(title=f"Episode {episode} - Prices", x_axis_label="t", y_axis_label=r"\(p / p_{max} [1]\)", x_axis_type="datetime", width=900, height=320)
for i, name in enumerate(PAIRS_):
    f0.line(ts, ps[:,i] / ps[:,i].max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
    # f0.line(ts, smooth(ps[:,i] / ps[:,i].max(), [item['t'] for item in info[episode]], tau=60*3), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f0 = bk.figure(title=f"Episode {episode} - Portfolio Fraction", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
f0.line(ts, Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(PAIRS_):
    f0.line(ts, vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f1 = bk.figure(title=f"Episode {episode} - Portfolio Value", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
r1 = f1.line(ts, Vs, line_width=2, legend_label="Total V")
r2 = f1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
bk.show(f1)

fig = bk.figure(title=f"Episode {episode} - Rewards", x_axis_label="t", y_axis_label=r"Reward \(\left(\log(\frac{V_{t+1}}{V_t})\right)\)", x_axis_type="datetime", width=900, height=320)
fig.scatter(ts, rewards[episode], size=2, color=Category10[10][4], legend_label="Reward")

returns = []
gamma = 0.999
G = 0.0
for r in reversed(rewards[episode]):
    G = r + gamma * G
    returns.insert(0, G)
fig.line(ts, returns, line_width=2, color=Category10[10][5], legend_label="Return")

hist, edges = torch.histogram(torch.tensor(rewards[episode]), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2
figh = bk.figure(title="Reward Distribution", width=300, height=320)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)
fig.legend.click_policy = "hide"

bk.show(bk.row(fig, figh))

In [ ]:
# Validation rollout and diagnostics


# Validate

In [ ]:
start = int(len(history) * 0.9)

agent.net.reset(1)
env.save_history = False

hist_s = []
hist_r = []
hist_i = []
hist_a = []
hist_logp = []

state = env.reset(history[start])
done = False
for elem in history[start + 1:]:
    action, log_prob = agent.act(state.to_tensor(), explore=False, grad_enabled=False)
    next_state, reward, done, info_t = env.step(action, data=elem)

    hist_s.append(state)
    hist_r.append(float(reward))
    hist_i.append(info_t)
    hist_a.append(int(action))
    hist_logp.append(float(log_prob))

    state = next_state
    if done:
        break

if len(hist_i) == 0:
    raise ValueError("Validation rollout produced no steps.")

asset_names = list(PAIRS_.keys())
ts = [datetime.fromtimestamp(item["t"]) for item in hist_i]

Vs = torch.tensor([item["V"] for item in hist_i], dtype=torch.float32)
Cs = torch.tensor([item["C"] for item in hist_i], dtype=torch.float32)
ps = torch.tensor([item["p"] for item in hist_i], dtype=torch.float32)
ws = torch.tensor([item["w"] for item in hist_i], dtype=torch.float32)
rewards_val = torch.tensor(hist_r, dtype=torch.float32)
log_probs_val = torch.tensor(hist_logp, dtype=torch.float32)

action_ids = torch.tensor(hist_a, dtype=torch.long)
action_types = torch.tensor([item["action_type"] for item in hist_i], dtype=torch.long)
action_assets = torch.tensor([item["action_asset"] for item in hist_i], dtype=torch.long)
action_fracs = torch.tensor([item["action_frac"] for item in hist_i], dtype=torch.float32)
valid_trades = torch.tensor([float(item["valid_trade"]) for item in hist_i], dtype=torch.float32)
realized_pnl = torch.tensor([sum(item["realized_pnl"]) for item in hist_i], dtype=torch.float32)
realized_cost = torch.tensor([sum(item["realized_cost"]) for item in hist_i], dtype=torch.float32)

portfolio_frac = ps * ws / Vs[:, None].clamp_min(1e-8)
cash_frac = Cs / Vs.clamp_min(1e-8)
cum_reward = rewards_val.cumsum(0)
cum_realized_pnl = realized_pnl.cumsum(0)
cum_realized_cost = realized_cost.cumsum(0)
running_peak = torch.cummax(Vs, dim=0).values
drawdown = 1.0 - Vs / running_peak.clamp_min(1e-8)
norm_prices = ps / ps[0].clamp_min(1e-8)
norm_value = Vs / Vs[0].clamp_min(1e-8)

buy_mask = action_types == 1
sell_mask = action_types == 2
hold_mask = action_types == 0
trade_mask = buy_mask | sell_mask
invalid_mask = (~hold_mask) & (valid_trades == 0)

cumulative_buys = buy_mask.to(torch.float32).cumsum(0)
cumulative_sells = sell_mask.to(torch.float32).cumsum(0)
cumulative_invalid = invalid_mask.to(torch.float32).cumsum(0)

validation_metrics = {
    "steps": len(hist_i),
    "terminated": bool(done),
    "final_value": float(Vs[-1].item()),
    "final_cash": float(Cs[-1].item()),
    "total_return_pct": float((norm_value[-1] - 1.0).item() * 100.0),
    "max_drawdown_pct": float(drawdown.max().item() * 100.0),
    "total_reward": float(cum_reward[-1].item()),
    "mean_reward": float(rewards_val.mean().item()),
    "reward_std": float(rewards_val.std(unbiased=False).item()),
    "buy_count": int(buy_mask.sum().item()),
    "sell_count": int(sell_mask.sum().item()),
    "hold_count": int(hold_mask.sum().item()),
    "invalid_trade_count": int(invalid_mask.sum().item()),
    "valid_trade_rate_pct": float(valid_trades.mean().item() * 100.0),
    "trade_rate_pct": float(trade_mask.to(torch.float32).mean().item() * 100.0),
    "avg_trade_fraction": float(action_fracs[trade_mask].mean().item()) if bool(trade_mask.any()) else 0.0,
    "realized_pnl_total": float(cum_realized_pnl[-1].item()),
    "realized_cost_total": float(cum_realized_cost[-1].item()),
}
validation_metrics


In [ ]:
for key, value in validation_metrics.items():
    if isinstance(value, float):
        print(f"{key:>22}: {value:.6f}")
    else:
        print(f"{key:>22}: {value}")


In [ ]:
overview_skip = max(1, len(ts) // 1500)

fig_prices = bk.figure(
    title="Validation - Normalized Prices vs Portfolio",
    width=1000,
    height=360,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Normalized Value [1]",
)
for i, name in enumerate(asset_names):
    fig_prices.line(ts[::overview_skip], norm_prices[::overview_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][i % 10])
fig_prices.line(ts[::overview_skip], norm_value[::overview_skip].numpy(), line_width=3, legend_label="Portfolio", color="black")
fig_prices.legend.click_policy = "hide"

fig_value = bk.figure(
    title="Validation - Portfolio Value and Cash",
    width=1000,
    height=320,
    x_range=fig_prices.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig_value.line(ts, Vs.numpy(), line_width=2, legend_label="Portfolio Value", color=Category10[10][0])
fig_value.line(ts, Cs.numpy(), line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][1])
fig_value.legend.location = "bottom_right"

fig_drawdown = bk.figure(
    title="Validation - Drawdown",
    width=1200,
    height=280,
    x_range=fig_prices.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Drawdown [%]",
)
fig_drawdown.line(ts, (100.0 * drawdown).numpy(), line_width=2, color=Category10[10][3], legend_label="Drawdown")
fig_drawdown.legend.location = "bottom_right"

bk.show(bk.column(fig_prices, fig_value, fig_drawdown))


In [ ]:
reward_skip = max(1, len(ts) // 2000)
reward_bins = min(200, max(20, len(rewards_val) // 10))

fig_reward = bk.figure(
    title="Validation - Step Reward",
    width=1200,
    height=280,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Reward",
)
fig_reward.line(ts[::reward_skip], rewards_val[::reward_skip].numpy(), line_width=2, color=Category10[10][4], legend_label="Step Reward")
fig_reward.legend.location = "bottom_right"

fig_cum_reward = bk.figure(
    title="Validation - Cumulative Reward",
    width=1200,
    height=280,
    x_range=fig_reward.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Cumulative Reward",
)
fig_cum_reward.line(ts, cum_reward.numpy(), line_width=2, color=Category10[10][2], legend_label="Cumulative Reward")
fig_cum_reward.legend.location = "bottom_right"

hist, edges = torch.histogram(rewards_val, bins=reward_bins, density=True)
x = ((edges[:-1] + edges[1:]) / 2).numpy()

fig_hist = bk.figure(
    title="Validation - Reward Distribution",
    width=380,
    height=320,
    x_axis_label="Density",
    y_axis_label="Reward",
)
fig_hist.harea(y=x, x1=0, x2=hist.numpy(), fill_color=Category10[10][4], fill_alpha=0.35)
fig_hist.line(hist.numpy(), x, line_color=Category10[10][4], line_width=2)

fig_realized = bk.figure(
    title="Validation - Cumulative Realized PnL and Cost",
    width=800,
    height=320,
    x_range=fig_reward.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig_realized.line(ts, cum_realized_pnl.numpy(), line_width=2, color=Category10[10][0], legend_label="Cumulative Realized PnL")
fig_realized.line(ts, cum_realized_cost.numpy(), line_width=2, color=Category10[10][1], line_dash="dashed", legend_label="Cumulative Realized Cost")
fig_realized.legend.location = "bottom_right"

bk.show(bk.column(fig_reward, fig_cum_reward, bk.row(fig_hist, fig_realized)))


In [ ]:
alloc_skip = max(1, len(ts) // 1500)

fig_alloc = bk.figure(
    title="Validation - Portfolio Fractions",
    width=1200,
    height=360,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Fraction [1]",
)
fig_alloc.line(ts[::alloc_skip], cash_frac[::alloc_skip].numpy(), line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(asset_names):
    fig_alloc.line(ts[::alloc_skip], portfolio_frac[::alloc_skip, i].numpy(), line_width=2, legend_label=name, color=Category10[10][(i + 1) % 10])
fig_alloc.legend.click_policy = "hide"

fig_actions = bk.figure(
    title="Validation - Cumulative Actions",
    width=1200,
    height=280,
    x_range=fig_alloc.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Count",
)
fig_actions.line(ts, cumulative_buys.numpy(), line_width=2, color=Category10[10][2], legend_label="Buys")
fig_actions.line(ts, cumulative_sells.numpy(), line_width=2, color=Category10[10][3], legend_label="Sells")
fig_actions.line(ts, cumulative_invalid.numpy(), line_width=2, color=Category10[10][5], legend_label="Invalid Trades")
fig_actions.legend.location = "bottom_right"

fig_trade_size = bk.figure(
    title="Validation - Trade Fractions",
    width=1200,
    height=280,
    x_range=fig_alloc.x_range,
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Action Fraction [1]",
    y_range=(0.0, 1.05),
)
if bool(buy_mask.any()):
    buy_idx = buy_mask.nonzero(as_tuple=False).squeeze(-1).tolist()
    fig_trade_size.scatter([ts[i] for i in buy_idx], action_fracs[buy_mask].numpy(), size=8, color=Category10[10][2], legend_label="Buy")
if bool(sell_mask.any()):
    sell_idx = sell_mask.nonzero(as_tuple=False).squeeze(-1).tolist()
    fig_trade_size.scatter([ts[i] for i in sell_idx], action_fracs[sell_mask].numpy(), size=8, color=Category10[10][3], legend_label="Sell")
fig_trade_size.legend.location = "bottom_right"

bk.show(bk.column(fig_alloc, fig_actions, fig_trade_size))
